In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
customers = pd.read_csv("customers.csv")
products = pd.read_csv("products.csv")
sales = pd.read_csv("sales.csv")
inventory = pd.read_csv("inventory.csv")

In [4]:
print("Customers:", customers.shape)
print("Products:", products.shape)
print("Sales:", sales.shape)
print("Inventory:", inventory.shape)

Customers: (1000, 7)
Products: (100, 6)
Sales: (10010, 7)
Inventory: (400, 4)


In [5]:
customers.head()

,Customer_ID,Customer_Name,Gender,Age,City,State,Region
0,C0001,Anjali Mehta,Female,55,Udaipur,Rajasthan,North
1,C0002,Rohit Nair,Male,42,Chennai,Tamil Nadu,South
2,C0003,Aarav Nair,Male,19,Patna,Bihar,East
3,C0004,Rahul More,Female,56,Amritsar,Punjab,North
4,C0005,Ananya Pawar,Male,24,Hyderabad,Telangana,South


In [6]:
products.head()

,Product_ID,Product_Name,Category,Sub_Category,Unit_Cost,Unit_Price
0,P0001,Pro Laptop 1,Electronics,Laptops,58322.43,76089.18
1,P0002,Pro Mobile 2,Electronics,Mobiles,16594.99,19957.15
2,P0003,Pro Accessorie 3,Electronics,Accessories,4852.77,5766.11
3,P0004,Pro Chair 4,Furniture,Chairs,4053.96,5422.13
4,P0005,Pro Table 5,Furniture,Tables,3602.57,4470.07


In [7]:
sales.head()

,Order_ID,Order_Date,Customer_ID,Product_ID,Region,Quantity,Discount
0,O000001,2025-09-12,C0649,P0032,East,4,0.00
1,O000002,2025-04-13,C0355,P0029,West,1,0.02
2,O000003,2025-01-14,C0916,P0063,East,5,0.02
3,O000004,2025-11-03,C0211,P0006,East,8,0.05
4,O000005,2026-07-23,C0658,P0090,North,2,0.00


In [8]:
inventory.head()

,Product_ID,Region,Stock_Quantity,Reorder_Level
0,P0001,West,157,33
1,P0001,South,7,22
2,P0001,North,106,14
3,P0001,East,70,15
4,P0002,West,190,31


In [9]:
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10010 entries, 0 to 10009
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Order_ID     10010 non-null  object 
 1   Order_Date   10010 non-null  object 
 2   Customer_ID  10010 non-null  object 
 3   Product_ID   10010 non-null  object 
 4   Region       10010 non-null  object 
 5   Quantity     10010 non-null  int64  
 6   Discount     9990 non-null   float64
dtypes: float64(1), int64(1), object(5)
memory usage: 547.6+ KB


In [10]:
sales.isnull().sum()

,0
Order_ID,0
Order_Date,0
Customer_ID,0
Product_ID,0
Region,0
Quantity,0
Discount,20


In [11]:
print("CUSTOMERS")
print(customers.isnull().sum())

print("\nPRODUCTS")
print(products.isnull().sum())

print("\nSALES")
print(sales.isnull().sum())

print("\nINVENTORY")
print(inventory.isnull().sum())

CUSTOMERS
Customer_ID      0
Customer_Name    0
Gender           0
Age              0
City             0
State            0
Region           0
dtype: int64

PRODUCTS
Product_ID      0
Product_Name    0
Category        0
Sub_Category    0
Unit_Cost       0
Unit_Price      0
dtype: int64

SALES
Order_ID        0
Order_Date      0
Customer_ID     0
Product_ID      0
Region          0
Quantity        0
Discount       20
dtype: int64

INVENTORY
Product_ID        0
Region            0
Stock_Quantity    0
Reorder_Level     0
dtype: int64


In [12]:
sales.duplicated().sum()

np.int64(10)

In [13]:
sales = sales.drop_duplicates()

In [14]:
sales.duplicated().sum()

np.int64(0)

In [15]:
sales.shape

(10000, 7)

Quantity = 0 records.

In [16]:
(sales["Quantity"] <= 0).sum()

np.int64(10)

Remove them:

In [17]:
sales = sales[sales["Quantity"] > 0]

now check


In [18]:
(sales["Quantity"] <= 0).sum()

np.int64(0)

HANDLE MISSING DISCOUNT

FIRST CHECK

In [19]:
sales["Discount"].isnull().sum()

np.int64(20)

So replace missing values with 0:

In [20]:
sales["Discount"] = sales["Discount"].fillna(0)

NOW CHECK

In [21]:
sales["Discount"].isnull().sum()

np.int64(0)

Convert Order_Date

In [22]:
sales["Order_Date"] = pd.to_datetime(sales["Order_Date"])

NOW Check:

In [23]:
sales.dtypes

,0
Order_ID,object
Order_Date,datetime64[ns]
Customer_ID,object
Product_ID,object
Region,object
Quantity,int64
Discount,float64


Merge Products with Sales

In [24]:
sales = sales.merge(
    products,
    on="Product_ID",
    how="left"
)

In [25]:
sales.head()

,Order_ID,Order_Date,Customer_ID,Product_ID,Region,Quantity,Discount,Product_Name,Category,Sub_Category,Unit_Cost,Unit_Price
0,O000001,2025-09-12,C0649,P0032,East,4,0.00,Classic Mobile 32,Electronics,Mobiles,39140.16,48337.53
1,O000002,2025-04-13,C0355,P0029,West,1,0.02,Plus Outdoor 29,Sports,Outdoor,5356.58,6584.10
2,O000003,2025-01-14,C0916,P0063,East,5,0.02,Smart Accessorie 63,Electronics,Accessories,1280.37,1532.68
3,O000004,2025-11-03,C0211,P0006,East,8,0.05,Pro Storage 6,Furniture,Storage,5485.98,6595.86
4,O000005,2026-07-23,C0658,P0090,North,2,0.00,Eco Sportswear 90,Sports,Sportswear,3174.68,3987.83


Calculate Sales

In [26]:
sales["Sales"] = (
    sales["Quantity"]
    * sales["Unit_Price"]
    * (1 - sales["Discount"])
)

Calculate Cost

In [27]:
sales["Cost"] = (
    sales["Quantity"]
    * sales["Unit_Cost"]
)

Calculate Profit

In [28]:
sales["Profit"] = (
    sales["Sales"]
    - sales["Cost"]
)

TO SEE THE ADDITIONAL COLUMN

In [29]:
sales[[
    "Product_ID",
    "Quantity",
    "Unit_Price",
    "Discount",
    "Sales",
    "Cost",
    "Profit"
]].head(10)

,Product_ID,Quantity,Unit_Price,Discount,Sales,Cost,Profit
0,P0032,4,48337.53,0.00,193350.1200,156560.64,36789.4800
1,P0029,1,6584.10,0.02,6452.4180,5356.58,1095.8380
2,P0063,5,1532.68,0.02,7510.1320,6401.85,1108.2820
3,P0006,8,6595.86,0.05,50128.5360,43887.84,6240.6960
4,P0090,2,3987.83,0.00,7975.6600,6349.36,1626.3000
5,P0027,4,5850.57,0.05,22232.1660,19515.44,2716.7260
6,P0099,3,624.06,0.10,1684.9620,1419.27,265.6920
7,P0028,4,2257.92,0.00,9031.6800,7942.88,1088.8000
8,P0005,5,4470.07,0.05,21232.8325,18012.85,3219.9825
9,P0041,2,998.36,0.05,1896.8840,1467.10,429.7840


**first business analysis**

**TOTAL SALES**

In [30]:
total_sales = sales["Sales"].sum()

print("Total Sales:", round(total_sales, 2))

Total Sales: 341966787.93


**TOTAL PROFIT**

In [31]:
total_profit = sales["Profit"].sum()

print("Total Profit:", round(total_profit, 2))

Total Profit: 52714801.6


**Profit Margin**

In [32]:
profit_margin = (total_profit / total_sales) * 100

print("Profit Margin:", round(profit_margin, 2), "%")

Profit Margin: 15.42 %


**Sales by category**

In [33]:
category_sales = (
    sales.groupby("Category")["Sales"]
    .sum()
    .sort_values(ascending=False)
)

category_sales

,Sales
Category,
Electronics,2.344750e+08
Furniture,4.851581e+07
Home Appliances,3.334716e+07
Sports,2.256558e+07
Office Supplies,3.063278e+06


**Sales by region**

In [34]:
region_sales = (
    sales.groupby("Region")["Sales"]
    .sum()
    .sort_values(ascending=False)
)

region_sales

,Sales
Region,
South,9.775366e+07
East,9.378370e+07
North,9.171679e+07
West,5.871263e+07


**Top 10 products**

In [35]:
top_products = (
    sales.groupby("Product_Name")["Sales"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top_products

,Sales
Product_Name,
Smart Laptop 61,2.772815e+07
Classic Laptop 31,2.683043e+07
Plus Laptop 16,2.128464e+07
Pro Laptop 1,2.079365e+07
Max Laptop 91,1.781055e+07
Classic Mobile 32,1.742858e+07
Eco Mobile 77,1.714174e+07
Max Mobile 92,1.507523e+07
Premium Laptop 46,1.486321e+07


**First save the cleaned sales file**

In [36]:
sales.to_csv("cleaned_sales.csv", index=False)

**Check that the file exists**

In [37]:
import os

os.listdir()

['.config',
 'inventory.csv',
 'customers.csv',
 'cleaned_sales.csv',
 'sales.csv',
 'products.csv',
 'sample_data']

**Save the other cleaned files**

In [38]:
customers.to_csv("cleaned_customers.csv", index=False)

products.to_csv("cleaned_products.csv", index=False)

inventory.to_csv("cleaned_inventory.csv", index=False)

In [39]:
import zipfile

files = [
    "cleaned_sales.csv",
    "cleaned_customers.csv",
    "cleaned_products.csv",
    "cleaned_inventory.csv"
]

with zipfile.ZipFile("cleaned_retail_data.zip", "w") as zipf:
    for file in files:
        zipf.write(file)

print("ZIP file created successfully!")

ZIP file created successfully!
